## Celda 1 - Checklist Conforme a los Archivos ```NPY```
Cargar ```.npy``` con Memmap

In [1]:
from pathlib import Path
import os, random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.regularizers import l2  # <-- NUEVO: para L2 regularization
from sklearn.metrics import classification_report, confusion_matrix

SEED = 123

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

BASE = Path(r"C:\Users\leona\Documents\Thesis_Project_UACH\Temp\Dataset\features_mfcc_labeled")

X_train = np.load(BASE / "X_train.npy", mmap_mode="r")
y_train = np.load(BASE / "y_train.npy")
X_val   = np.load(BASE / "X_val.npy",   mmap_mode="r")
y_val   = np.load(BASE / "y_val.npy")
X_test  = np.load(BASE / "X_test.npy",  mmap_mode="r")
y_test  = np.load(BASE / "y_test.npy")

print("X_train:", X_train.shape, X_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("X_val:  ", X_val.shape, X_val.dtype)
print("X_test: ", X_test.shape, X_test.dtype)

# Chequeo de clases
u, c = np.unique(y_train, return_counts=True)
print("Distribución train:", dict(zip(u, c)))

X_train: (52551, 3, 32, 201) float16
y_train: (52551,) int64
X_val:   (11253, 3, 32, 201) float16
X_test:  (11308, 3, 32, 201) float16
Distribución train: {np.int64(0): np.int64(7368), np.int64(1): np.int64(6535), np.int64(2): np.int64(10668), np.int64(3): np.int64(27980)}


### Celda 3.2 - ```tf.data``` (con cast a float32 dentro del pipeline)

In [2]:
BATCH = 64

def make_ds(X, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if training:
        ds = ds.shuffle(20000, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH)
    ds = ds.map(lambda a,b: (tf.cast(a, tf.float32), b), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_ds(X_train, y_train, training=True)
val_ds   = make_ds(X_val, y_val, training=False)
test_ds  = make_ds(X_test, y_test, training=False)

### Celda 3.3 - Class weights (para el desbalance)

Esto ayuda mucho para que no “adivine todo clase 3” (clase predominante).

In [3]:
num_classes = 4
counts = np.bincount(y_train, minlength=num_classes)
total = counts.sum()

# Peso inverso a frecuencia (simple y efectivo)
class_weight = {i: float(total / (num_classes * counts[i])) for i in range(num_classes) if counts[i] > 0}
print("counts:", counts)
print("class_weight:", class_weight)

counts: [ 7368  6535 10668 27980]
class_weight: {0: 1.7830822475570032, 1: 2.0103672532517214, 2: 1.2315101237345332, 3: 0.4695407433881344}


### Celda 3.4 - Modelo CNN baseline (channels_first)

In [4]:
input_shape = (3, 32, 201)
act_fn = "sigmoid"

# ===== REGULARIZACIÓN L2 (weight decay ligero) =====
l2_reg = 1.5e-4  # Ajustar: 5e-5 para más ligero, 2e-4 para más fuerte

model = models.Sequential([
    layers.Input(shape=input_shape),
    
    # Bloque 1: Conv + BN + Pool
    layers.Conv2D(16, (3,3), padding="same", activation=act_fn, 
                  data_format="channels_first", kernel_regularizer=l2(l2_reg)),
    layers.BatchNormalization(axis=1),
    layers.MaxPool2D((2,2), data_format="channels_first"),
    
    # Bloque 2: Conv + BN + Pool
    layers.Conv2D(32, (3,3), padding="same", activation=act_fn, 
                  data_format="channels_first", kernel_regularizer=l2(l2_reg)),
    layers.BatchNormalization(axis=1),
    layers.MaxPool2D((2,2), data_format="channels_first"),
    layers.Dropout(0.25),
    
    # Bloque 3: Conv + BN + Pool
    layers.Conv2D(64, (3,3), padding="same", activation=act_fn, 
                  data_format="channels_first", kernel_regularizer=l2(l2_reg)),
    layers.BatchNormalization(axis=1),
    layers.MaxPool2D((2,2), data_format="channels_first"),
    
    # Bloque 4: Conv + BN + GlobalAvgPool
    layers.Conv2D(128, (3,3), padding="same", activation=act_fn, 
                  data_format="channels_first", kernel_regularizer=l2(l2_reg)),
    layers.BatchNormalization(axis=1),
    layers.GlobalAveragePooling2D(data_format="channels_first"),
    
    # Clasificador con regularización
    layers.Dense(64, activation="relu", kernel_regularizer=l2(l2_reg)),
    layers.Dropout(0.35),  # Dropout moderado antes de softmax
    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(7.5e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 16, 32, 201)    │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 16, 32, 201)    │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 100)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 16, 100)    │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 16, 100)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 8, 50)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32, 8, 50)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 8, 50)      │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64, 8, 50)      │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 64, 4, 25)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 128, 4, 25)     │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 128, 4, 25)     │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 106,916 (417.64 KB)

 Trainable params: 106,436 (415.77 KB)

 Non-trainable params: 480 (1.88 KB)

### Celda 3.5 - Entrenar (con callbacks)

In [5]:
# ===== CALLBACKS OPTIMIZADOS PARA REDUCIR OVERFITTING =====
callbacks = [
    # Detiene si val_loss no mejora en 6 épocas (más sensible que 10)
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",          # <-- Cambiado de val_accuracy a val_loss
        mode="min",                   # <-- min porque queremos loss bajo
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    # Reduce LR si val_loss se estanca
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",          # <-- Cambiado de val_accuracy a val_loss
        mode="min",
        patience=5,
        factor=0.5,
        min_lr=1e-6,                 # <-- evita LR demasiado pequeño
        verbose=1
    ),
    # Guarda el mejor modelo según val_accuracy (para métricas finales)
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(BASE / "cnn_mfcc_queenRec_regularized.h5"),
        monitor="val_accuracy",      # Guardamos el mejor por accuracy
        mode="max",
        save_best_only=True,
        verbose=1
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    class_weight=class_weight,
    callbacks=callbacks
)

Epoch 1/40
821/822 ━━━━━━━━━━━━━━━━━━━━ 0s 271ms/step - accuracy: 0.6985 - loss: 0.8237
Epoch 1: val_accuracy improved from None to 0.76424, saving model to C:\Users\leona\Documents\Thesis_Project_UACH\Temp\Dataset\features_mfcc_labeled\cnn_mfcc_queenRec_regularized.h5


822/822 ━━━━━━━━━━━━━━━━━━━━ 242s 286ms/step - accuracy: 0.7603 - loss: 0.6819 - val_accuracy: 0.7642 - val_loss: 0.8982 - learning_rate: 7.5000e-04
Epoch 2/40
821/822 ━━━━━━━━━━━━━━━━━━━━ 0s 272ms/step - accuracy: 0.8593 - loss: 0.4247
Epoch 2: val_accuracy improved from 0.76424 to 0.83862, saving model to C:\Users\leona\Documents\Thesis_Project_UACH\Temp\Dataset\features_mfcc_labeled\cnn_mfcc_queenRec_regularized.h5


822/822 ━━━━━━━━━━━━━━━━━━━━ 234s 284ms/step - accuracy: 0.8634 - loss: 0.4214 - val_accuracy: 0.8386 - val_loss: 0.6396 - learning_rate: 7.5000e-04
Epoch 3/40
821/822 ━━━━━━━━━━━━━━━━━━━━ 0s 269ms/step - accuracy: 0.8868 - loss: 0.3458
Epoch 3: val_accuracy did not improve from 0.83862
822/822 ━━━━━━━━━━━━━━━━━━━━ 232s 282ms/step - accuracy: 0.8894 - loss: 0.3476 - val_accuracy: 0.8322 - val_loss: 0.5964 - learning_rate: 7.5000e-04
Epoch 4/40
821/822 ━━━━━━━━━━━━━━━━━━━━ 0s 269ms/step - accuracy: 0.9074 - loss: 0.2940
Epoch 4: val_accuracy did not improve from 0.83862
822/822 ━━━━━━━━━━━━━━━━━━━━ 231s 281ms/step - accuracy: 0.9067 - loss: 0.3049 - val_accuracy: 0.8086 - val_loss: 0.6386 - learning_rate: 7.5000e-04
Epoch 5/40
821/822 ━━━━━━━━━━━━━━━━━━━━ 0s 269ms/step - accuracy: 0.9190 - loss: 0.2631
Epoch 5: val_accuracy improved from 0.83862 to 0.84457, saving model to C:\Users\leona\Documents\Thesis_Project_UACH\Temp\Dataset\features_mfcc_labeled\cnn_mfcc_queenRec_regularized.h5


822/822 ━━━━━━━━━━━━━━━━━━━━ 232s 282ms/step - accuracy: 0.9198 - loss: 0.2699 - val_accuracy: 0.8446 - val_loss: 0.6143 - learning_rate: 7.5000e-04
Epoch 6/40
821/822 ━━━━━━━━━━━━━━━━━━━━ 0s 271ms/step - accuracy: 0.9286 - loss: 0.2442
Epoch 6: val_accuracy did not improve from 0.84457
822/822 ━━━━━━━━━━━━━━━━━━━━ 233s 284ms/step - accuracy: 0.9259 - loss: 0.2555 - val_accuracy: 0.7967 - val_loss: 0.7516 - learning_rate: 7.5000e-04
Epoch 7/40
821/822 ━━━━━━━━━━━━━━━━━━━━ 0s 271ms/step - accuracy: 0.9309 - loss: 0.2376
Epoch 7: val_accuracy improved from 0.84457 to 0.85853, saving model to C:\Users\leona\Documents\Thesis_Project_UACH\Temp\Dataset\features_mfcc_labeled\cnn_mfcc_queenRec_regularized.h5


822/822 ━━━━━━━━━━━━━━━━━━━━ 233s 283ms/step - accuracy: 0.9324 - loss: 0.2412 - val_accuracy: 0.8585 - val_loss: 0.6124 - learning_rate: 7.5000e-04
Epoch 8/40
821/822 ━━━━━━━━━━━━━━━━━━━━ 0s 270ms/step - accuracy: 0.9342 - loss: 0.2216
Epoch 8: ReduceLROnPlateau reducing learning rate to 0.000375000003259629.

Epoch 8: val_accuracy did not improve from 0.85853
822/822 ━━━━━━━━━━━━━━━━━━━━ 232s 282ms/step - accuracy: 0.9356 - loss: 0.2289 - val_accuracy: 0.7919 - val_loss: 1.0931 - learning_rate: 7.5000e-04
Epoch 9/40
821/822 ━━━━━━━━━━━━━━━━━━━━ 0s 269ms/step - accuracy: 0.9499 - loss: 0.1857
Epoch 9: val_accuracy did not improve from 0.85853
822/822 ━━━━━━━━━━━━━━━━━━━━ 231s 281ms/step - accuracy: 0.9511 - loss: 0.1862 - val_accuracy: 0.8560 - val_loss: 0.6588 - learning_rate: 3.7500e-04
Epoch 10/40
821/822 ━━━━━━━━━━━━━━━━━━━━ 0s 269ms/step - accuracy: 0.9583 - loss: 0.1664
Epoch 10: val_accuracy did not improve from 0.85853
822/822 ━━━━━━━━━━━━━━━━━━━━ 231s 281ms/step - accuracy: 0

822/822 ━━━━━━━━━━━━━━━━━━━━ 230s 280ms/step - accuracy: 0.9612 - loss: 0.1603 - val_accuracy: 0.8635 - val_loss: 0.6546 - learning_rate: 3.7500e-04
Epoch 13: early stopping
Restoring model weights from the end of the best epoch: 3.


### Celda 3.6 - Evaluación en Test (matriz + macro-F1)

In [6]:
probs = model.predict(test_ds)
y_pred = np.argmax(probs, axis=1)

print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred, digits=4))

177/177 ━━━━━━━━━━━━━━━━━━━━ 13s 66ms/step
Confusion matrix:
 [[ 834  113  236  408]
 [   8  927  181  297]
 [  29  133 1697  440]
 [   2  167  267 5569]]

Classification report:
               precision    recall  f1-score   support

           0     0.9553    0.5242    0.6769      1591
           1     0.6918    0.6561    0.6734      1413
           2     0.7127    0.7381    0.7252      2299
           3     0.8295    0.9274    0.8757      6005

    accuracy                         0.7983     11308
   macro avg     0.7973    0.7114    0.7378     11308
weighted avg     0.8062    0.7983    0.7919     11308

